# Query-Level Failure Case Studies

## 1. Imports, Paths, Constants

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths 
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE10_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase10_fold_robustness'
PHASE11_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase11_case_studies'
PHASE11_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLD='Fold1'
BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
N_EXAMPLES=3  #Number of example queries

print('='*80)
print('PHASE 11: QUERY-LEVEL FAILURE CASE STUDIES')
print('='*80)
print(f'Output: {PHASE11_OUTPUT}')
print(f'Dataset: MQ{DATASET} {FOLD}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print(f'N examples: {N_EXAMPLES}')
print('\nPurpose: Illustrate structural patterns from Phases 8-10')
print('='*80)

PHASE 11: QUERY-LEVEL FAILURE CASE STUDIES
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase11_case_studies
Dataset: MQ2007 Fold1
Baseline: pointwise_raw
N examples: 3

Purpose: Illustrate structural patterns from Phases 8-10


## 2. Loading Phase 6 Baseline Artifacts

In [2]:
print('\n'+'='*80)
print('LOADING BASELINE ARTIFACTS')
print('='*80)

baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
qm_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_query_metrics.csv'
pred_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_predictions.csv'

if not qm_file.exists():
    raise RuntimeError(f'Missing: {qm_file}')
if not pred_file.exists():
    raise RuntimeError(f'Missing: {pred_file}')

qm=pd.read_csv(qm_file)
pred=pd.read_csv(pred_file)

#Validating
for col in ['qid', 'num_relevant_1', 'Failure@5_primary']:
    if col not in qm.columns:
        raise RuntimeError(f'Missing column: {col}')

for col in ['qid', 'label', 'score']:
    if col not in pred.columns:
        raise RuntimeError(f'Missing column: {col}')

print(f'Query metrics: {len(qm)} queries')
print(f'Predictions: {len(pred)} documents')
print('='*80)


LOADING BASELINE ARTIFACTS
Query metrics: 336 queries
Predictions: 13652 documents


## 3. Loading Persistent Queries from Phase 10

In [3]:
print('\n'+'='*80)
print('LOADING PERSISTENT QUERIES FROM PHASE 10')
print('='*80)
print('Using persistent queries already identified in Phase 10')
print('Persistent = intersection of failures across ALL 9 configs')
print('='*80)

#Phase 10 doesn't store qid lists directly, so we reconstruct from baseline
#but only within the persistent set identified in Phase 10

#Loading all 9 configs to identify persistent set
models=['pointwise', 'pairwise', 'lightgbm']
pipelines=['raw', 'global', 'per_query']

def make_fail_flag(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).round().astype(int).clip(0, 1)
    return series.map(lambda v: 1 if str(v).lower() in ['true','1','yes'] else 0).astype(int)

evaluable_sets={}
failure_sets={}

for model in models:
    for pipeline in pipelines:
        key=f'{model}_{pipeline}_{DATASET}'
        qm_file=PHASE6_OUTPUT / FOLD / f'{key}_query_metrics.csv'
        
        if not qm_file.exists():
            print(f'Missing: {key}')
            continue
        
        qm_config=pd.read_csv(qm_file)
        evaluable_sets[key]=set(qm_config.loc[qm_config['num_relevant_1'] > 0, 'qid'])
        
        fail_flag=make_fail_flag(qm_config['Failure@5_primary'])
        fail_mask=(qm_config['num_relevant_1'] > 0) & (fail_flag == 1)
        failure_sets[key]=set(qm_config.loc[fail_mask, 'qid'])

#Computing persistent (same logic as Phase 10)
if len(evaluable_sets)==9 and len(failure_sets)==9:
    all_evaluable=set.intersection(*evaluable_sets.values())
    persistent_qids=set.intersection(*failure_sets.values())
    print(f'\nLoaded {len(evaluable_sets)} configs')
    print(f'Evaluable queries: {len(all_evaluable)}')
    print(f'Persistent failures: {len(persistent_qids)}')
else:
    raise RuntimeError(f'Expected 9 configs, found {len(evaluable_sets)}')

print('\nPersistent = queries failing across ALL 9 model×pipeline configs')
print('This matches the definition from Phase 10')
print('='*80)


LOADING PERSISTENT QUERIES FROM PHASE 10
Using persistent queries already identified in Phase 10
Persistent = intersection of failures across ALL 9 configs

Loaded 9 configs
Evaluable queries: 290
Persistent failures: 22

Persistent = queries failing across ALL 9 model×pipeline configs
This matches the definition from Phase 10


## 4. Selecting Example Queries

In [4]:
print('\n'+'='*80)
print('SELECTING EXAMPLE QUERIES')
print('=' * 80)

#Computing score_gap for persistent queries
def compute_score_gap(qid):
    q_docs=pred[pred['qid'] == qid].copy()
    if len(q_docs)==0:
        return np.nan
    q_docs=q_docs.sort_values('score', ascending=False)
    relevant=q_docs[q_docs['label']>=1]
    if len(relevant)==0:
        return np.nan
    best_rel=float(relevant['score'].max())
    k_actual=min(5, len(q_docs))
    if k_actual==0:
        return np.nan
    score_at_5=float(q_docs.iloc[k_actual-1]['score'])
    return best_rel - score_at_5

#Building candidate data for persistent queries
candidate_data=[]
for qid in persistent_qids:
    gap=compute_score_gap(qid)
    qm_row=qm[qm['qid'] == qid]
    if len(qm_row)==0:
        continue
    qm_row=qm_row.iloc[0]
    candidate_data.append({
        'qid': qid,
        'num_relevant_1': int(qm_row['num_relevant_1']),
        'score_gap': gap
    })

candidate_df=pd.DataFrame(candidate_data).dropna(subset=['score_gap'])

#Selecting queries that clearly demonstrate structural failure patterns:
# - Extreme sparsity (num_relevant_1 = 1)
# - Weak score separation (small score_gap)
print('\nSelection criteria:')
print('num_relevant_1 = 1 (extreme sparsity)')
print('score_gap <= 0.02 (weak separation)')

candidate_df=candidate_df[
    (candidate_df['num_relevant_1']==1) &
    (candidate_df['score_gap']<=0.02)
]

#Sorting by score_gap (most negative first)
candidate_df=candidate_df.sort_values('score_gap')

print(f'\nFound {len(candidate_df)} candidates meeting criteria')

if len(candidate_df) < N_EXAMPLES:
    print(f'Only {len(candidate_df)} candidates, using all')
    selected_qids=candidate_df['qid'].tolist()
else:
    selected_qids=candidate_df.head(N_EXAMPLES)['qid'].tolist()

print(f'\nSelected {len(selected_qids)} example queries:')
for qid in selected_qids:
    row=candidate_df[candidate_df['qid']==qid].iloc[0]
    print(f"QID {qid}: num_rel={row['num_relevant_1']}, gap={row['score_gap']:.4f}")

print('\nThese queries illustrate the structural patterns from Phase 8:')
print('Extreme sparsity -> high failure risk')
print('Weak score separation -> marginal failures')
print('='*80)


SELECTING EXAMPLE QUERIES

Selection criteria:
num_relevant_1 = 1 (extreme sparsity)
score_gap <= 0.02 (weak separation)

Found 11 candidates meeting criteria

Selected 3 example queries:
QID 8352: num_rel=1.0, gap=-0.3126
QID 9991: num_rel=1.0, gap=-0.2975
QID 8342: num_rel=1.0, gap=-0.2556

These queries illustrate the structural patterns from Phase 8:
Extreme sparsity -> high failure risk
Weak score separation -> marginal failures


## 5. Generating Query Summaries and Rankings

In [5]:
print('\n'+'='*80)
print('GENERATING QUERY SUMMARIES AND RANKINGS')
print('='*80)

query_summaries=[]
all_rankings=[]

for qid in selected_qids:
    print(f'\nQuery {qid}:')
    
    #Getting query metrics
    qm_row=qm[qm['qid']==qid].iloc[0]
    num_rel=int(qm_row['num_relevant_1'])
    fail_flag=make_fail_flag(pd.Series([qm_row['Failure@5_primary']]))[0]
    gap=compute_score_gap(qid)
    
    query_summaries.append({
        'qid': qid,
        'num_relevant_1': num_rel,
        'score_gap': float(gap) if not np.isnan(gap) else None,
        'failure_at_5': 'Yes' if fail_flag == 1 else 'No'
    })
    
    print(f'num_relevant_1: {num_rel}')
    print(f'score_gap: {gap:.4f}')
    print(f'Failure@5: {"Yes" if fail_flag==1 else "No"}')
    
    #Getting top-10 ranking
    q_docs=pred[pred['qid']==qid].copy()
    q_docs=q_docs.sort_values('score', ascending=False).head(10)
    q_docs['rank']=range(1, len(q_docs) + 1)
    q_docs['relevant']=(q_docs['label'] >= 1).astype(int)
    
    for _, doc in q_docs.iterrows():
        all_rankings.append({
            'qid': qid,
            'rank': int(doc['rank']),
            'score': float(doc['score']),
            'label': int(doc['label']),
            'relevant': int(doc['relevant'])
        })
    
    #Showing relevant positions
    rel_ranks=q_docs[q_docs['relevant']==1]['rank'].tolist()
    if rel_ranks:
        print(f'Relevant at ranks: {rel_ranks}')
        if all(r > 5 for r in rel_ranks):
            print('-> All relevant docs BELOW rank-5 cutoff')
    else:
        print('No relevant in top-10')

#Saving summaries
summary_df=pd.DataFrame(query_summaries)
summary_df.to_csv(PHASE11_OUTPUT / 'phase11_case_study_queries.csv', index=False)
print('\nSaved: phase11_case_study_queries.csv')

#Saving rankings
rankings_df=pd.DataFrame(all_rankings)
rankings_df.to_csv(PHASE11_OUTPUT / 'phase11_case_study_rankings.csv', index=False)
print('Saved: phase11_case_study_rankings.csv')

print('='*80)


GENERATING QUERY SUMMARIES AND RANKINGS

Query 8352:
num_relevant_1: 1
score_gap: -0.3126
Failure@5: Yes
No relevant in top-10

Query 9991:
num_relevant_1: 1
score_gap: -0.2975
Failure@5: Yes
No relevant in top-10

Query 8342:
num_relevant_1: 1
score_gap: -0.2556
Failure@5: Yes
No relevant in top-10

Saved: phase11_case_study_queries.csv
Saved: phase11_case_study_rankings.csv


## 6. Generating Visualizations

In [6]:
print('\n'+'='*80)
print('GENERATING VISUALIZATIONS')
print('='*80)

for qid in selected_qids:
    print(f'\nPlotting QID {qid}...')
    
    #Getting data
    q_rankings=rankings_df[rankings_df['qid']==qid].sort_values('rank')
    q_summary=summary_df[summary_df['qid']==qid].iloc[0]
    
    #Creating figure
    fig, ax=plt.subplots(figsize=(12, 6))
    
    #Plotting all scores
    ranks=q_rankings['rank'].values
    scores=q_rankings['score'].values
    relevant=q_rankings['relevant'].values
    
    #Non-relevant docs
    non_rel_mask=relevant==0
    ax.scatter(ranks[non_rel_mask], scores[non_rel_mask], 
               s=100, alpha=0.6, label='Non-relevant', marker='o')
    
    #Relevant docs
    rel_mask=relevant==1
    ax.scatter(ranks[rel_mask], scores[rel_mask],
               s=250, alpha=0.9, label='Relevant', marker='*', 
               edgecolors='black', linewidths=2.5)
    
    #Annotating relevant documents
    for rank, score, is_rel in zip(ranks, scores, relevant):
        if is_rel==1:
            ax.annotate(
                'Relevant',
                xy=(rank, score),
                xytext=(10, 10),
                textcoords='offset points',
                fontsize=10,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1.5)
            )
    
    #Line connecting scores
    ax.plot(ranks, scores, alpha=0.3, linestyle='--', color='gray', linewidth=1.5)
    
    #Rank-5 cutoff line
    ax.axvline(x=5.5, color='red', linestyle='--', linewidth=2.5, alpha=0.7, label='Rank-5 cutoff')
    
    #Labels and title
    ax.set_xlabel('Rank Position', fontsize=13, fontweight='bold')
    ax.set_ylabel('Ranking Score', fontsize=13, fontweight='bold')
    title=f'Query {qid} — Persistent Failure Example\n'
    title+=f'num_relevant={q_summary["num_relevant_1"]}, '
    title+=f'score_gap={q_summary["score_gap"]:.4f}, '
    title+=f'Failure@5={q_summary["failure_at_5"]}'
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
    ax.grid(alpha=0.3, linestyle=':')
    ax.set_xticks(range(1, 11))
    
    plt.tight_layout()
    plt.savefig(PHASE11_OUTPUT / f'phase11_case_study_plot_qid_{qid}.png', 
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: phase11_case_study_plot_qid_{qid}.png')

print('\nAll visualizations saved')
print('='*80)


GENERATING VISUALIZATIONS

Plotting QID 8352...
Saved: phase11_case_study_plot_qid_8352.png

Plotting QID 9991...
Saved: phase11_case_study_plot_qid_9991.png

Plotting QID 8342...
Saved: phase11_case_study_plot_qid_8342.png

All visualizations saved


## 7. Case Study Representatives Check

In [ ]:
print('\n'+'='*80)
print('CASE STUDY REPRESENTATIVENESS CHECK')
print('='*80)

#Computing metrics for ALL persistent queries
all_persistent_data=[]

for qid in persistent_qids:
    gap=compute_score_gap(qid)
    qm_row=qm[qm['qid']==qid]
    if len(qm_row)==0:
        continue
        
    qm_row=qm_row.iloc[0]
    
    all_persistent_data.append({
        'qid': qid,
        'num_relevant_1': int(qm_row['num_relevant_1']),
        'score_gap': gap
    })

all_df=pd.DataFrame(all_persistent_data).dropna()

#Metrics for selected examples
example_df=all_df[all_df['qid'].isin(selected_qids)]

summary=pd.DataFrame({
    'group': ['all_persistent', 'case_studies'],
    'n_queries': [len(all_df), len(example_df)],
    'median_num_relevant_1': [
        all_df['num_relevant_1'].median(),
        example_df['num_relevant_1'].median()
    ],
    'median_score_gap': [
        all_df['score_gap'].median(),
        example_df['score_gap'].median()
    ]
})

display(summary)

summary.to_csv(
    PHASE11_OUTPUT / 'phase11_case_study_representativeness.csv',
    index=False
)

print('\nSaved: phase11_case_study_representativeness.csv')
print('='*80)